# 0~1단계: Safebooru 수집 + Phash 테스트 (VS Code 로컬용)

VS Code에서 이 파일을 열고 오른쪽 위에서 Python 커널을 선택한 뒤 `모두 실행`을 누르세요. GPU는 필요 없습니다.

- **0단계**: Safebooru API로 이미지와 태그를 수집
- **1단계**: Phash로 중복 제거 + "유사 이미지 검색에 쓸 수 있는가" 확인

이미지와 결과 파일은 이 노트북과 같은 폴더의 `drawing_ref_test/`에 저장됩니다.

> 수집한 이미지는 작가들의 저작물입니다. 팀 내부 기술 검증용으로만 쓰고 외부에 재배포하지 마세요. Git에 올린다면 `drawing_ref_test/`를 `.gitignore`에 추가하세요.

In [ ]:
%pip install -q imagehash pillow requests pandas tqdm matplotlib ipywidgets

## 설정
여기 값만 바꾸면 됩니다.

In [ ]:
SEARCH_TAGS = "solo full_body"   # Safebooru 검색 태그 (공백으로 구분, 영어)
NUM_IMAGES  = 500                # 수집할 장수
MAX_SIDE    = 1024               # 저장 시 긴 변 최대 픽셀 (용량 절약)

# Safebooru가 인증을 요구할 경우에만 입력 (계정 > Options 페이지에서 확인)
API_KEY = ""
USER_ID = ""

import os
BASE = os.path.abspath("./drawing_ref_test")   # 노트북과 같은 폴더 아래에 저장
IMG_DIR = os.path.join(BASE, "images")
os.makedirs(IMG_DIR, exist_ok=True)
print("저장 위치:", BASE)

## 0단계-1. 메타데이터(태그, 이미지 주소) 수집

In [ ]:
import requests, time, pandas as pd

API = "https://safebooru.org/index.php"
HEADERS = {"User-Agent": "graduation-project-test/0.1"}

def fetch_page(pid, limit=100):
    params = {"page": "dapi", "s": "post", "q": "index", "json": 1,
              "limit": limit, "pid": pid, "tags": SEARCH_TAGS}
    if API_KEY and USER_ID:
        params.update({"api_key": API_KEY, "user_id": USER_ID})
    r = requests.get(API, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    if not r.text.strip():
        return []                      # 결과가 없으면 빈 응답이 옵니다
    try:
        data = r.json()
    except ValueError:
        raise RuntimeError("JSON이 아닌 응답입니다. 인증이 필요할 수 있어요:\n" + r.text[:300])
    if isinstance(data, dict):
        data = data.get("post", [])
    return data

posts, pid = [], 0
while len(posts) < NUM_IMAGES:
    page = fetch_page(pid)
    if not page:
        break
    posts.extend(page)
    pid += 1
    time.sleep(1)                      # 서버 부담을 줄이기 위한 대기
posts = posts[:NUM_IMAGES]

def image_url(p):
    if p.get("file_url"):
        return p["file_url"]
    return f"https://safebooru.org/images/{p['directory']}/{p['image']}"

meta = pd.DataFrame([{
    "id": p["id"], "url": image_url(p), "tags": p.get("tags", ""),
    "width": p.get("width"), "height": p.get("height"),
    "source_page": f"https://safebooru.org/index.php?page=post&s=view&id={p['id']}",
} for p in posts])
print(len(meta), "건 수집")
meta.head()

## 0단계-2. 이미지 다운로드

In [ ]:
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

paths, failed = [], 0
for row in tqdm(meta.itertuples(), total=len(meta)):
    path = os.path.join(IMG_DIR, f"{row.id}.jpg")
    if not os.path.exists(path):
        try:
            r = requests.get(row.url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img.thumbnail((MAX_SIDE, MAX_SIDE))
            img.save(path, quality=90)
            time.sleep(0.3)
        except Exception as e:
            failed += 1
            path = None
    paths.append(path)

meta["path"] = paths
meta = meta.dropna(subset=["path"]).reset_index(drop=True)
meta.to_csv(os.path.join(BASE, "metadata.csv"), index=False)
print(f"저장 {len(meta)}장 / 실패 {failed}장")

## 1단계-1. Phash 계산

In [ ]:
import imagehash, numpy as np

hashes = [imagehash.phash(Image.open(p)) for p in tqdm(meta["path"])]
meta["phash"] = [str(h) for h in hashes]

# 64비트 해시를 0/1 배열로 바꿔 모든 쌍의 해밍 거리(0=동일, 64=완전히 다름)를 계산
bits = np.array([h.hash.flatten() for h in hashes], dtype=np.uint8)
dist = (bits[:, None, :] != bits[None, :, :]).sum(axis=2)
print("거리 행렬:", dist.shape)

## 1단계-2. 중복 이미지 찾기
해밍 거리가 `DUP_THRESHOLD` 이하면 같은 그림으로 봅니다. 결과를 눈으로 보고 값을 조정하세요.

In [ ]:
import matplotlib.pyplot as plt
DUP_THRESHOLD = 6

n = len(meta)
pairs = [(i, j, int(dist[i, j])) for i in range(n) for j in range(i + 1, n)
         if dist[i, j] <= DUP_THRESHOLD]
print(f"중복 의심 쌍: {len(pairs)}개")

for i, j, d in pairs[:10]:             # 앞 10쌍만 표시
    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    for a, k in zip(ax, (i, j)):
        a.imshow(Image.open(meta.path[k])); a.axis("off"); a.set_title(str(meta.id[k]))
    fig.suptitle(f"hamming distance = {d}")
    plt.show()

In [ ]:
# 중복 쌍에서 뒤쪽 이미지를 제외한 정리본 저장
drop = sorted({j for _, j, _ in pairs})
clean = meta.drop(index=drop).reset_index(drop=True)
clean.to_csv(os.path.join(BASE, "metadata_dedup.csv"), index=False)
print(f"{len(meta)}장 → 중복 제거 후 {len(clean)}장")

## 1단계-3. Phash로 '비슷한 그림'이 찾아지는가?
임의의 이미지를 골라 Phash 거리가 가장 가까운 8장을 봅니다. 맨 왼쪽이 검색 이미지입니다. 구도/포즈가 비슷한 그림이 나오는지 팀이 직접 판단하세요.

In [ ]:
import random
TOP_K = 8

def show_similar(q):
    order = [k for k in np.argsort(dist[q]) if k != q][:TOP_K]
    fig, ax = plt.subplots(1, TOP_K + 1, figsize=(2.2 * (TOP_K + 1), 3))
    ax[0].imshow(Image.open(meta.path[q])); ax[0].set_title("QUERY"); ax[0].axis("off")
    for a, k in zip(ax[1:], order):
        a.imshow(Image.open(meta.path[k])); a.set_title(f"d={dist[q, k]}"); a.axis("off")
    plt.show()

for q in random.sample(range(len(meta)), 5):
    show_similar(q)

## 1단계-4. 거리 분포
대부분의 쌍이 거리 25~35 부근에 몰려 있으면, Phash가 '다른 그림들 사이의 유사도'를 구분하지 못한다는 뜻입니다.

In [ ]:
tri = dist[np.triu_indices(len(meta), k=1)]
plt.hist(tri, bins=range(0, 66, 2))
plt.xlabel("hamming distance"); plt.ylabel("pairs"); plt.title("Phash distance distribution")
plt.show()
print(f"평균 {tri.mean():.1f} / 거리 {DUP_THRESHOLD} 이하 비율 {(tri <= DUP_THRESHOLD).mean() * 100:.3f}%")

## 결과 기록 (팀이 작성)

| 항목 | 결과 |
| --- | --- |
| 수집 장수 / 실패 장수 | |
| 중복 의심 쌍 수 (임계값) | |
| 중복 판정이 실제로 맞았는가 | |
| 유사 검색 결과에서 구도·포즈가 비슷했던 비율 (5개 쿼리 × 8장) | |
| 결론: Phash의 용도 | |

다음 단계(2단계)는 같은 이미지 폴더와 `metadata_dedup.csv`를 그대로 사용합니다.